In [1]:
import torch
import torch.nn.functional as F
from torch import nn
import torchvision
from torchvision import datasets, models, transforms
from torchsummary import summary

import gc
import re
import time
import math
import random
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as patches
import numpy as np 
import os
os.environ["CUDA_VISIBLE_DEVICES"]="0"

from copy import deepcopy

# import sys
# sys.path.append(os.path.dirname(os.path.abspath(os.path.dirname('__file__'))))

import setting
from utils import *
from custom_dataset import cub
from explainer import vgg_lrps
from explainer.lrp_utils import *


In [2]:
model_name = 'vgg16' 
# model_name = 'resnet50'
dataset_name = 'imagenet'

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

LOG_DIR = 'logs'
ICP_DIR = 'icp'

log_model_data_path = os.path.join(LOG_DIR, '{}/{}'.format(model_name, 
                                                           dataset_name))
# train_log_path = os.path.join(log_model_data_path, '{}_{}_log.txt'.format(model_name,
#                                                                           dataset_name))
icp_path = os.path.join(ICP_DIR, 'icp_image_4_5_7_11_14.npy')


if dataset_name == 'cub':
    dataset_root = '/archive/workspace/datasets/CUB_200_2011'
    class_txt = os.path.join(dataset_root, 'classes.txt')
    data_dir = os.path.join(dataset_root, 'split')
    
    with open(class_txt, 'r') as f:
        lines = f.readlines()
    
    class_names = {}
    for line in lines:
        cls_idx = int(line.split(' ')[0])-1
        cls_name = line.split('.')[1].replace('\n', '')
        class_names[cls_idx] = cls_name
    nb_classes = len(class_names.keys())
    input_size = 224
    
elif dataset_name == 'imagenet':
    data_dir = '/archive/workspace/datasets/ILSVRC2012'
    nb_classes = 1000
    input_size = 224

In [3]:
model = torchvision.models.vgg16(pretrained='IMAGENET1K_V1')
model.eval()
model = model.to(device)
summary(model, (3, 224, 224))

/archive/library/anaconda3/envs/xai/lib/python3.7/site-packages/torchvision/models/_utils.py:209: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and will be removed in 0.15, please use 'weights' instead.
  f"The parameter '{pretrained_param}' is deprecated since 0.13 and will be removed in 0.15, "
/archive/library/anaconda3/envs/xai/lib/python3.7/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and will be removed in 0.15. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 64, 224, 224]           1,792
              ReLU-2         [-1, 64, 224, 224]               0
            Conv2d-3         [-1, 64, 224, 224]          36,928
              ReLU-4         [-1, 64, 224, 224]               0
         MaxPool2d-5         [-1, 64, 112, 112]               0
            Conv2d-6        [-1, 128, 112, 112]          73,856
              ReLU-7        [-1, 128, 112, 112]               0
            Conv2d-8        [-1, 128, 112, 112]         147,584
              ReLU-9        [-1, 128, 112, 112]               0
        MaxPool2d-10          [-1, 128, 56, 56]               0
           Conv2d-11          [-1, 256, 56, 56]         295,168
             ReLU-12          [-1, 256, 56, 56]               0
           Conv2d-13          [-1, 256, 56, 56]         590,080
             ReLU-14          [-1, 256,

In [4]:
norm_factor_mean = [0.485, 0.456, 0.406]
norm_factor_std = [0.229, 0.224, 0.225]

train_ds = datasets.ImageFolder(
    os.path.join(data_dir, 'train'), 
    torchvision.transforms.Compose([
        torchvision.transforms.Resize(size=(input_size, input_size)),
        torchvision.transforms.ToTensor(),
        torchvision.transforms.Normalize(mean=norm_factor_mean, std=norm_factor_std)
        # transforms.RandomResizedCrop(input_size),
        # transforms.RandomHorizontalFlip(),
        # transforms.ToTensor(),
        # transforms.Normalize(mean=norm_factor_mean, std=norm_factor_std)
    ]))

train_dl = torch.utils.data.DataLoader(train_ds,
                                       # batch_size=setting.sz_batch_train, 
                                       batch_size=80, 
                                       shuffle=True, 
                                       num_workers=4)

val_ds = datasets.ImageFolder(
    os.path.join(data_dir, 'val'), 
    torchvision.transforms.Compose([
        torchvision.transforms.Resize(size=(input_size, input_size)),
        torchvision.transforms.ToTensor(),
        torchvision.transforms.Normalize(mean=norm_factor_mean, std=norm_factor_std)
        # transforms.Resize(int(input_size/0.875)),
        # transforms.CenterCrop(input_size),
        # transforms.ToTensor(),
        # transforms.Normalize(mean=norm_factor_mean, std=norm_factor_std)
    ]))

val_dl = torch.utils.data.DataLoader(val_ds,
                                     batch_size=10, 
                                     shuffle=True, 
                                     num_workers=4)

steps_of_train = int(len(train_ds)/setting.sz_batch_train)
steps_of_val = int(len(val_ds)/10)

print('setting.sz_batch_train): ', setting.sz_batch_train)
print('steps_of_train: ', steps_of_train)
print('setting.sz_batch_val): ', 10)
print('steps_of_val: ', steps_of_val)

setting.sz_batch_train):  80
steps_of_train:  16014
setting.sz_batch_val):  10
steps_of_val:  5000


In [5]:
# gc.collect()
# torch.cuda.empty_cache()

# lrp = vgg_lrps.VGG_LRP(name=model_name, model=model, device=device, 
#               input_size=input_size, nb_classes=nb_classes)

In [6]:
# for i, (images, labels) in enumerate(val_dl):  
#     images = images.to(device)
#     labels = labels.to(device)
    
#     conv_idx = lrp.check_pt
#     # conv_idx.insert(0, 0)
#     print('conv_idx: ', conv_idx)
    
#     target_idx = conv_idx[0]
#     s_idx = -1
#     t_idx = 0
    
#     r = None
#     for idx in reversed(conv_idx):
#         if r == None: r = images
#         else: s_idx = t_idx-1
#         t_idx = idx
        
#         r, a, p = lrp.relevance(r=r, y=None, s_idx=s_idx, t_idx=t_idx)
#         if idx == target_idx: break
    
#     break

In [7]:
# col = 2
# row = r.shape[0]
# uf = 4

# fig, axs = plt.subplots(row, col, figsize=(col*uf, row*uf))

# for b in range(row):
#     for i in range(col):
#         x = int(i/col) + b
#         y = int(i%col) 
        
#         if i == 0:
#             img = images[b].permute(1,2,0).squeeze().detach().cpu()
#             axs[x,y].imshow(img)
            
#         else: 
#             hm = r[b].permute(1,2,0).max(dim=-1)[0].detach().cpu()
#             axs[x,y].imshow(hm, vmin=-hm.max(), vmax=hm.max(), cmap='seismic')


In [8]:
if os.path.isfile(icp_path):
    print('Load icp_dict')
    icp_dict = np.load(icp_path, allow_pickle=True).item()
else:
    print('Generate icp_dict')
    icp_dict = generate_icp(sz_patches=setting.sz_patches_vgg_imgnet_core, 
                            nb_classes=nb_classes)
    np.save(icp_path, icp_dict) 

Load icp_dict


In [9]:
# layer_info = model_info['layer']
# layer_info_keys = list(layer_info.keys())
# print(layer_info_keys)
# print(layer_info['features'])
# x = torch.zeros([1, 3, input_size, input_size]).to(device)

# activations = []
# with torch.no_grad():
#     activations.append(torch.ones_like(x))
#     for key in layer_info_keys:
#         print('key: ', key)
#         for layer in layer_info[key].values():
#             print(layer)
#             x = layer.forward(x)
#             print('x: ', x.shape)
#             activations.append(x)

In [10]:
def load_hyperparameters(conv_idx, conv_shape, model_name, layer_sep_type):
    nb_attrs = {}
    sz_patches = {}
    sz_attrs = {}
    strength = {}
    alpha = {}
    beta = {}
    lr = {}
    
    print(len(conv_idx))
    
    for i, (idx, shape) in enumerate(zip(conv_idx, conv_shape)):
        # nb_attrs
        if setting.nb_attrs == 'MANUAL': 
            if model_name == 'vgg16' and layer_sep_type == setting.LAYER_SEP_TYPE_FULL:
                nb_attrs[idx] = setting.nb_attrs_vgg16_full[i]
            elif model_name == 'vgg16' and layer_sep_type == setting.LAYER_SEP_TYPE_CORE:
                nb_attrs[idx] = setting.nb_attrs_vgg16_core[i]
            elif model_name == 'resnet50' and layer_sep_type == setting.LAYER_SEP_TYPE_FULL:
                nb_attrs[idx] = setting.nb_attrs_resnet50[i]
            elif model_name == 'resnet50' and layer_sep_type == setting.LAYER_SEP_TYPE_CORE:
                nb_attrs[idx] = setting.nb_attrs_resnet50_core[i]

        # sz_patches
        if setting.sz_patches == 'MANUAL': 
            if model_name == 'vgg16' and layer_sep_type == setting.LAYER_SEP_TYPE_FULL:
                sz_patches[idx] = setting.sz_patches_vgg16_cub_full[i]
            elif model_name == 'vgg16' and layer_sep_type == setting.LAYER_SEP_TYPE_CORE:
                sz_patches[idx] = setting.sz_patches_vgg16_imagenet_core[i]
            elif model_name == 'resnet50' and layer_sep_type == setting.LAYER_SEP_TYPE_FULL:
                sz_patches[idx] = setting.sz_patches_resnet50_cub[i]
            elif model_name == 'resnet50' and layer_sep_type == setting.LAYER_SEP_TYPE_CORE:
                sz_patches[idx] = setting.sz_patches_resnet50_imagenet_core[i]
        else:
            sz_patches[idx] = int(shape[-1]/nb_attrs[idx])

        # sz_attrs
        if setting.sz_attrs == 'MANUAL': 
            if model_name == 'vgg16' and layer_sep_type == setting.LAYER_SEP_TYPE_FULL:
                sz_attrs[idx] = setting.sz_attrs_vgg16_cub_full[i]
            elif model_name == 'vgg16' and layer_sep_type == setting.LAYER_SEP_TYPE_CORE:
                sz_attrs[idx] = setting.sz_attrs_vgg16_imagenet_core[i]
            elif model_name == 'resnet50' and layer_sep_type == setting.LAYER_SEP_TYPE_FULL:
                sz_attrs[idx] = setting.sz_attrs_resnet50_cub[i]
            elif model_name == 'resnet50' and layer_sep_type == setting.LAYER_SEP_TYPE_CORE:
                sz_attrs[idx] = setting.sz_attrs_resnet50_imagenet_core[i]
        else:
            sz_attrs[idx] = int((sz_patches[idx]-1)/2)

        # strength
        if setting.strength == 'MANUAL': 
            if model_name == 'vgg16' and layer_sep_type == setting.LAYER_SEP_TYPE_FULL:
                strength[idx] = setting.strength_vgg16_full[i]
            elif model_name == 'vgg16' and layer_sep_type == setting.LAYER_SEP_TYPE_CORE:
                strength[idx] = setting.strength_vgg16_core[i]
            elif model_name == 'resnet50' and layer_sep_type == setting.LAYER_SEP_TYPE_FULL:
                strength[idx] = setting.strength_resnet50[i]
            elif model_name == 'resnet50' and layer_sep_type == setting.LAYER_SEP_TYPE_CORE:
                strength[idx] = setting.strength_resnet50_core[i]

        # alpah 
        if setting.alpha == 'MANUAL': 
            if model_name == 'vgg16' and layer_sep_type == setting.LAYER_SEP_TYPE_FULL:
                alpha[idx] = setting.alpha_vgg16_full[i]
            elif model_name == 'vgg16' and layer_sep_type == setting.LAYER_SEP_TYPE_CORE:
                alpha[idx] = setting.alpha_vgg16_core[i]
            elif model_name == 'resnet50' and layer_sep_type == setting.LAYER_SEP_TYPE_FULL:
                alpha[idx] = setting.alpha_resnet50[i]
            elif model_name == 'resnet50' and layer_sep_type == setting.LAYER_SEP_TYPE_CORE:
                alpha[idx] = setting.alpha_resnet50_core[i]

        # beta
        if setting.beta == 'MANUAL': 
            if model_name == 'vgg16' and layer_sep_type == setting.LAYER_SEP_TYPE_FULL:
                beta[idx] = setting.beta_vgg16_full[i]
            elif model_name == 'vgg16' and layer_sep_type == setting.LAYER_SEP_TYPE_CORE:
                beta[idx] = setting.beta_vgg16_core[i]
            elif model_name == 'resnet50' and layer_sep_type == setting.LAYER_SEP_TYPE_FULL:
                beta[idx] = setting.beta_resnet50[i]
            elif model_name == 'resnet50' and layer_sep_type == setting.LAYER_SEP_TYPE_CORE:
                beta[idx] = setting.beta_resnet50_core[i]

        # lr
        if setting.lr == 'MANUAL': 
            if model_name == 'vgg16':
                lr[idx] = setting.lr_vgg16_full[i]
        else:
            lr[idx] =  0.01 * (sz_attrs[idx]/2)

    print('nb_attrs: ', nb_attrs)
    print('sz_patches: ', sz_patches)
    print('sz_attrs: ', sz_attrs)
    print('strength: ', strength)
    print('alpha: ', alpha)
    print('beta: ', beta)
    print('lr: ', lr)

    return nb_attrs, sz_patches, sz_attrs, strength, alpha, beta, lr
        

In [11]:
# Class-wise Layer-wise Attribute Model 
class CLAM(nn.Module):
    def __init__(self, 
                 model, 
                 device, 
                 icp_dict,
                 model_name: str=None, 
                 explainer_name: str=setting.NAME_OF_EXPLAINER_LRP,
                 layer_sep_type: str='FULL', 
                 nb_classes: int=0):
        super().__init__()
        
        self.model = model
        self.model.eval()
        self.device = device
        self.icp_dict = icp_dict
        self.model_name = model_name
        self.explainer_name = explainer_name
        self.layer_sep_type = layer_sep_type
        self.nb_classes = nb_classes
        
        self.explainer = self._init_explainer(exp_name=self.explainer_name)
        
        self.conv_idx, self.cla_layers, self.nb_attrs, self.sz_patches, \
            self.sz_attrs, self.lr, self.max_epoches, self.strength = self._init_cla_layers()

        self.nb_epochs = self.max_epoches * len(self.cla_layers.keys())
        
    def _init_explainer(self, exp_name):
        if exp_name == setting.NAME_OF_EXPLAINER_LRP:
            if self.model_name == 'vgg16':
                explainer = vgg_lrps.VGG_LRP(name=self.model_name, model=self.model, 
                                             device=self.device, input_size=input_size, 
                                             nb_classes=self.nb_classes, pt_range=self.layer_sep_type)
            elif self.model_name == 'resnet50':
                print('load ResNet_LRP')
                explainer = resnet_lrps.ResNet_LRP(name=self.model_name, model=self.model, 
                                                   device=self.device, input_size=input_size, 
                                                   nb_classes=self.nb_classes, pt_range=self.layer_sep_type)
            
        elif exp_name == setting.NAME_OF_EXPLAINER_CLRP:
            if self.model_name == 'vgg16':
                explainer = vgg_lrps.VGG_CLRP(name=self.model_name, model=self.model, 
                                             device=self.device, input_size=input_size, 
                                             nb_classes=self.nb_classes, pt_range=self.layer_sep_type)
            elif self.model_name == 'resnet50':
                print('load ResNet_LRP')
                explainer = resnet_lrps.ResNet_CLRP(name=self.model_name, model=self.model, 
                                                    device=self.device, input_size=input_size, 
                                                    nb_classes=self.nb_classes, pt_range=self.layer_sep_type)
            
        elif exp_name == setting.NAME_OF_EXPLAINER_SGLRP:
            if self.model_name == 'vgg16':
                explainer = vgg_lrps.VGG_SGLRP(name=self.model_name, model=self.model, 
                                             device=self.device, input_size=input_size, 
                                             nb_classes=self.nb_classes, pt_range=self.layer_sep_type)
            elif self.model_name == 'resnet50':
                print('load ResNet_LRP')
                explainer = resnet_lrps.ResNet_SGLRP(name=self.model_name, model=self.model,
                                                     device=self.device, input_size=input_size, 
                                                     nb_classes=self.nb_classes, pt_range=self.layer_sep_type)
            
        else:
            print("Invalid explainer name, exiting...")
            exit()
            
        return explainer
        
    def _init_cla_layers(self):
        cla_layers = {}
        
        conv_idx = self.explainer.check_pt
        conv_shapes = self.explainer.shapes

        # max_epoches = setting.max_epoches
        max_epoches = 10
        nb_attrs, sz_patches, sz_attrs, \
            strength, alpha, beta, lr = load_hyperparameters(conv_idx=conv_idx, 
                                                             conv_shape=conv_shapes, 
                                                             model_name=self.model_name, 
                                                             layer_sep_type=self.layer_sep_type)
        
        for i, idx in enumerate(conv_idx):
            cla_layers[idx] = CLA_Layer(device=self.device, 
                                        idx=idx, 
                                        layer_shape=conv_shapes[idx], 
                                        nb_classes=self.nb_classes, 
                                        nb_attrs=nb_attrs[idx],
                                        sz_patch=sz_patches[idx], 
                                        sz_attr=sz_attrs[idx], 
                                        max_epoches=max_epoches, 
                                        alpha=alpha[idx], 
                                        beta=beta[idx], 
                                        lr=lr[idx])

        return conv_idx, cla_layers, nb_attrs, sz_patches, sz_attrs, lr, max_epoches, strength
    
    def _get_icp_as_ts(self, sz_p): 
        # (nb_classes, 1, sz_p, sz_p)
        return torch.tensor(self.icp_dict[sz_p])[:, None, :, :].cuda()
    
    def _get_rand_xy_exclusive_of_t_list(self, t_list, max_val):
        rand = np.zeros(t_list.shape, dtype=int)
        
        for k, l in enumerate(t_list): 
            nb_list_items = len(l)
            while(True):
                rand_xy = [random.randint(0, max_val) for i in range(nb_list_items)]
                checker = [rand_xy[j] == l[j]  for j in range(nb_list_items)]
                if True not in checker: 
                    rand[k] = rand_xy
                    break 
                    
        return rand
    
    def _get_r_max_idx(self, r, idx):
        r_2d = r.max(dim=1)[0]
        r_2d = F.avg_pool2d(input=r_2d, 
                            kernel_size=(self.sz_patches[idx], 
                                         self.sz_patches[idx]), 
                            stride=1)
        
        b, rw, rh = r_2d.shape
        k = self.nb_attrs[idx]
        r_1d = r_2d.reshape([b, rw*rh]) #.clone().detach()
        
        r_topk_val, r_topk_idx = torch.topk(r_1d, k=k, dim=1)
        
        # r_topk_idx -> row, col 나눔
        r_topk_row = np.zeros(r_topk_idx.shape).astype(np.int32)
        r_topk_col = np.zeros(r_topk_idx.shape).astype(np.int32)
        for i, idx in enumerate(r_topk_idx):
            r_topk_row[i] = idx.cpu().numpy()//rw
            r_topk_col[i] = idx.cpu().numpy()%rw
        
        return r_topk_row, r_topk_col
    
    def _generate_pos_neg(self, r, a, y, idx):
        """
            Generate pos/neg inputs for training cla_layers
            
            Args:
                r: Relevance (B, C, W, H)
                a: Activation (B, C, W, H)
                y: Label (B, nb_classes)
                idx: Current conv idx
                
           Returns: 
               f_pos: Pos sub feature (nb_attrs, B, C, sz_patch, sz_patch)
               f_neg: Neg sub feature (nb_attrs, B, C, sz_patch, sz_patch)
               r_topk_row: Top-k row index of relevance (k == nb_attr)
               r_topk_col: Top-k col index of relevance (k == nb_attr)
               
        """
        r_topk_row, r_topk_col = self._get_r_max_idx(r, idx)
        
        r2d_max_idx = r.shape[-1] - self.sz_patches[idx]
        r_row_rand_idx = self._get_rand_xy_exclusive_of_t_list(t_list=r_topk_row, 
                                                               max_val=r2d_max_idx)
        r_col_rand_idx = self._get_rand_xy_exclusive_of_t_list(t_list=r_topk_col,
                                                               max_val=r2d_max_idx)
        
        # (nb_attr, b, d, p, p)
        f_pos = torch.zeros([self.nb_attrs[idx], a.shape[0], a.shape[1], 
                             self.sz_patches[idx], self.sz_patches[idx]]).to(device)
        f_neg = torch.zeros([self.nb_attrs[idx], a.shape[0], a.shape[1], 
                             self.sz_patches[idx], self.sz_patches[idx]]).to(device) 
        
        r_topk_row = r_topk_row.T
        r_topk_col = r_topk_col.T
        r_row_rand_idx = r_row_rand_idx.T
        r_col_rand_idx = r_col_rand_idx.T
        
        
        # Extract sub_feature using top-k index for pos and rand index for neg
        for i in range(self.nb_attrs[idx]):
            indices = []
            for j, a_b in enumerate(a):
                f_pos[i,j] = a_b[:, r_topk_row[i][j]:r_topk_row[i][j]+self.sz_patches[idx],
                             r_topk_col[i][j]:r_topk_col[i][j]+self.sz_patches[idx]]
                f_neg[i,j] = a_b[:, r_row_rand_idx[i][j]:r_row_rand_idx[i][j]+self.sz_patches[idx],
                             r_col_rand_idx[i][j]:r_col_rand_idx[i][j]+self.sz_patches[idx]]
        
        # Add icp to last channel in pos/neg
        icp = torch.stack([self._get_icp_as_ts(self.sz_patches[idx]) 
                           for i in range(self.nb_attrs[idx])], dim=0)
        
        f_pos = torch.cat((f_pos, icp[:, y, :]), dim=2)
        
        rand_p = self._get_rand_xy_exclusive_of_t_list(t_list=y.unsqueeze(1), 
                                                       max_val=self.nb_classes-1)
        rand_p = torch.from_numpy(rand_p).squeeze()
        f_neg = torch.cat((f_neg, icp[:, rand_p, :]), dim=2)
        
        return f_pos, f_neg, r_topk_row, r_topk_col
    
    def _get_mask_for_strength(self, idx, mode='train'):
        if mode == 'train':
            s = self.strength[idx]
        else:
            s = 1.1 # for test 
        p = self.sz_patches[idx] 
        mask = np.zeros([p*2-1, p*2-1])
        w, h = mask.shape
        cp_x, cp_y = w//2, h//2
        
        for i in range(w):
            for j in range(h):        
                dis_x = np.abs(cp_x-i)
                dis_y = np.abs(cp_y-j)
                max_dis = dis_x if dis_x-dis_y>0 else dis_y

                if dis_x==0 and dis_y==0:
                    mask[i, j] = s
                else:
                    for k in range(1, p):
                        if max_dis==k:
                            str_v = s*(0.95**k)
                            if str_v < 1.0:
                                str_v = 1.0
                            mask[i, j] = str_v
                            break
        
        return torch.from_numpy(mask).to(device)
    
    def _get_new_pos(self, base_pos, w, h, idx, mask):
        m_w, m_h = mask.shape
        
        if base_pos-self.sz_patches[idx]+1 < 0: 
            s_pt = 0
            m_s_pt = -(base_pos-self.sz_patches[idx]+1)
        else:
            s_pt = base_pos-self.sz_patches[idx]+1
            m_s_pt = 0
        
        if base_pos+self.sz_patches[idx] > w:
            e_pt = w
            m_e_pt = w - base_pos + (m_w-self.sz_patches[idx])
        else: 
            e_pt = base_pos+self.sz_patches[idx]
            m_e_pt = m_w

        return s_pt, e_pt, m_s_pt, m_e_pt
    
    def _apply_min_dist_to_r(self, r, xy, idx, r_idx=None, mode='train'):
        mask = self._get_mask_for_strength(idx, mode) # (p-a+1, p-a+1)
        
        _, _, w, h = r.shape # r: (B, C, W, H)
        for i, r_b in enumerate(r):
            if r_idx != None:
                r_idx_topk = r_idx[i]  # k==nb_attrs[idx]
            else:
                r_idx_topk = [[0, 0] for i in range(self.nb_attrs[idx])]
                
            for j in range(xy.shape[1]):
                r_x, r_y = r_idx_topk[j]
                d_x, d_y = xy[i, j]
                
                x_start, x_end, mx_s_pt, mx_e_pt = self._get_new_pos(r_x+d_x, w, h, idx, mask.clone().detach())
                y_start, y_end, my_s_pt, my_e_pt = self._get_new_pos(r_y+d_y, w, h, idx, mask.clone().detach())

                r_b[:, x_start:x_end, y_start:y_end] *= mask[mx_s_pt:mx_e_pt, my_s_pt:my_e_pt]
                        
        return r
    
    
    def _combine_r_and_dist(self, r, dist_list, idx, r_topk_row, r_topk_col):
        """
            Generate new relevance map with dist_list 
        
        """
        sz_batch = r.shape[0]
        min_dist_xy = np.zeros([sz_batch, self.nb_attrs[idx], 2], dtype=int) # 2: row, col val
        for i, dist in enumerate(dist_list):
            b, a, w, h = dist.shape  # (b, nb_attrs, p-a+1, p-a+1)
            dist = dist.reshape(b, a, w*h)
            min_dist_pos = dist.argmin(dim=-1)
            min_dist_pos = min_dist_pos[:, i]
            for j, pos in enumerate(min_dist_pos):
                min_dist_xy[j, i, 0] = pos//w
                min_dist_xy[j, i, 1] = pos%h
        
        r_topk_row = r_topk_row.T
        r_topk_col = r_topk_col.T
        
        r_idx = []
        for i in range(sz_batch):
            r_idx_b = [[r_topk_row[i, j], r_topk_col[i, j]] for j in range(self.nb_attrs[idx])]
            r_idx.append(r_idx_b)
         
        return self._apply_min_dist_to_r(r, min_dist_xy, idx, r_idx)
            
    def _get_pre_r_and_a(self, x, y, target_idx):
        r = None
        dist_pos_list = None
        pred = None
        
        s_idx = -1
        t_idx = 0
        
        for idx in reversed(self.conv_idx):
            if r == None: r = x
            else: s_idx = t_idx-1
            t_idx = self.cla_layers[idx].idx
            
            r, a, p  = self.explainer.relevance(r=r, y=y, s_idx=s_idx, t_idx=t_idx)
            if pred == None: pred = p
                
            if idx == target_idx: break
            else:
                with torch.no_grad():
                    f_pos, _, r_topk_row, r_topk_col = self._generate_pos_neg(r, a, y, idx)
                    dist_pos_list = [self.cla_layers[idx](f_p) for f_p in f_pos]
                    r = self._combine_r_and_dist(r, dist_pos_list, idx, r_topk_row, r_topk_col)
                    
        return r, a, pred
    
    def _get_attribute_mask(self, xy, r_shape, idx):
        """
            For test
            # Generate a mask for min_dist point used for relevance refinement 
        """
        b, _, w, h = r_shape
        
        m = torch.ones([b, 1, w, h]).to(device)
        # if idx in self.conv_idx:
        m = self._apply_min_dist_to_r(r=m, xy=xy, idx=idx, mode='test')
        m -= 1.0
        
        return m.clone().detach().cpu()
    
    def get_origal_rs(self, x, y=None, target_idx=-1):
        x = x.to(self.device)
        if y is not None: y = y.to(self.device)
        
        r = None
        rest_a = None
        rs = []
        
        s_idx = -1
        t_idx = 0
        
        # for propagating to input layer, add idx -1
        
        conv_idx_expand = self.conv_idx.copy()  
        if target_idx == -1:
            conv_idx_expand.insert(0, -1) # for input layer 
        
        for idx in reversed(conv_idx_expand):
            if r == None: r = x
            else: s_idx = t_idx-1
            
            if idx == -1: t_idx = 0
            else: t_idx = self.cla_layers[idx].idx
            
            r, a, _, = self.explainer.relevance(r=r, y=y, s_idx=s_idx, t_idx=t_idx)
            rs.insert(0, r.clone().detach().cpu())
            
            if idx == target_idx: break
            
        return rs
                
    def train(self, start_attr_idx=-1, target_idx=0):
        keys = self.conv_idx
        print('keys: ', keys)
        
        # === for test
        if start_attr_idx >= 0:
            keys = keys[:start_attr_idx+1]
        print('target_idx: ', target_idx)
        
        train_log_path = os.path.join(log_model_data_path, 'ckpt_{}/log.txt'.format(self.explainer_name))
        print('train_log_path: ', train_log_path)
        with open(train_log_path, 'w') as f:
            f.write('')
        
        dist_pos_neg_diff = {k: 0. for k in keys}
        max_stopping_cnt = 1
        
        for i, idx in enumerate(reversed(keys)):
            print('idx: ', idx)
            attr_d, attr_w, attr_h = self.cla_layers[idx].attr_shape[1:]
            min_diff_mean = attr_d * attr_w * attr_h * self.nb_attrs[idx]
            early_stopping_cnt = 0
            
            for epoch in range(self.max_epoches):
                
                dist_pos_neg_diff[idx] = 0.
                
                for step, (x_batch, y_batch) in enumerate(train_dl): 
                    x_batch = x_batch.to(device)
                    y_batch = y_batch.to(device)
                    
                    r, a, pred = self._get_pre_r_and_a(x=x_batch, 
                                                       y=y_batch,
                                                       target_idx=idx)
                    
                    f_pos, f_neg, _, _ = self._generate_pos_neg(r=r,
                                                                a=a,
                                                                y=y_batch, 
                                                                idx=idx)
                    
                    
                    loss, pos_mean, neg_mean = self.cla_layers[idx].train(y=y_batch, 
                                                                          p=pred, 
                                                                          f_pos_list=f_pos,
                                                                          f_neg_list=f_neg)
                    
                    dist_pos_neg_diff[idx] += pos_mean.item() - neg_mean.item()
                    
                    
                    # if step % int(steps_of_train/2) == 0 and step > 0:
                    if step % int(steps_of_train/100) == 0 and step > 0:
                        p_output = 'step: {} in epoch {} (idx: {})\n'.format(step, epoch, idx)
                        p_output += 'lr: {}\n'.format(self.cla_layers[idx].opt.param_groups[0]['lr'])
                        p_output += '\tloss {} \n\tpos_mean {:.4f} (amass: {:.4f}) \n\tneg_mean {:.4f}\n'.format(loss.item(), 
                                                                                                                 pos_mean.item(),
                                                                                                                 dist_pos_neg_diff[idx]/step,
                                                                                                                 neg_mean.item())
                        print(p_output)
                        with open(train_log_path, 'a') as f:
                            f.write(p_output)  
                            
                cur_idx_dist_diff_mean = dist_pos_neg_diff[idx] / steps_of_train
                if cur_idx_dist_diff_mean < min_diff_mean:
                    min_diff_mean = cur_idx_dist_diff_mean
                    early_stopping_cnt = 0

                    print('Save model!')
                    torch.save({'epoch': epoch,
                                'layer_state_dict': self.cla_layers[idx].state_dict(),
                                'optstate_dict': self.cla_layers[idx].opt.state_dict(),
                                'dist_pos': dist_pos_neg_diff[idx] / step
                               }, '{}/ckpt_{}/{}.pth'.format(log_model_data_path, 
                                                             self.explainer_name,
                                                             idx))
                    self.cla_layers[idx].scheduler.step()

                else: 
                    early_stopping_cnt += 1

                e_output = self._print_output_epoch(epoch, dist_pos_neg_diff)
                # if early_stopping_cnt == setting.max_stopping_cnt: 
                if early_stopping_cnt == max_stopping_cnt: 
                    e_output += 'Skip to next epoch \n'
                    gc.collect()
                    torch.cuda.empty_cache()
                    break
                else:
                    e_output += 'Current stopping_cnt: {}\n\n'.format(early_stopping_cnt)
                print('e_output: ', e_output)
                with open(train_log_path, 'a') as f:
                    f.write(e_output)
                    
                # if early_stopping_cnt == setting.max_stopping_cnt: 
                if early_stopping_cnt == max_stopping_cnt: 
                    gc.collect()
                    torch.cuda.empty_cache()
                    break
                    
            if idx == target_idx: break
                
        print("FINISH")
        
    def get_mask(self, x, y=None, target_idx=-1):
        """
            Propagate CLA-layers on x batch
            
            Args: 
                x: Input image batch; (B, C, W, H)
                y: Label; (B, nb_classes), If label is None, prediction will be used as label 
                target_idx: Target conv idx for stopping propagation
            Returns:
                r: Generated relevance combining with mask predicted by attributes
                r_orig: Original relevance on input 
        """
        x = x.to(self.device)
        if y is not None: y = y.to(self.device)

        r = None
        pred = None
        r_attr = []
        acts = []
        attr_xy = []
        attr_map = []
        
        s_idx = -1
        t_idx = 0
        
        # for propagating to input layer, add idx -1
        conv_idx_expand = self.conv_idx.copy()
        if target_idx == -1:
            conv_idx_expand.insert(0, -1)
        
        for idx in reversed(conv_idx_expand):
            if r == None: r = x.clone().detach()
            else: s_idx = t_idx-1
            
            if idx == -1: t_idx = 0
            else: t_idx = self.cla_layers[idx].idx
            
            r, a, p = self.explainer.relevance(r=r, y=y, s_idx=s_idx, t_idx=t_idx)
            
            if pred == None: pred = p
            
            if idx == -1: 
                r_attr.insert(0, r.clone().detach().cpu())
                attr_xy.insert(0, 0)
                attr_map.insert(0, 0)
                acts.insert(0, a.clone().detach().cpu())
                break
                
            else:            
                with torch.no_grad():
                    attr_dist_xy, attr_dist_map = self.cla_layers[idx].predict(a=a,
                                                                               p=pred[1], 
                                                                               y=y, 
                                                                               icp=self._get_icp_as_ts(self.sz_patches[idx]))
                    
                    r = self._apply_min_dist_to_r(r, attr_dist_xy, idx, mode='test')
                    
                    r_attr.insert(0, r.clone().detach().cpu())
                    attr_xy.insert(0, attr_dist_xy)
                    attr_map.insert(0, attr_dist_map)
                    acts.insert(0, a.clone().detach().cpu())
            
            gc.collect()
            torch.cuda.empty_cache()

            if idx == target_idx: break
        
        return r_attr, attr_xy, attr_map, acts, pred
    
    def push_attributes(self, trainset_dl, x, y=None, target_idx=0):
        """
            Push each attribute in each layer to the nearest patch in the training set
            
            Args:
                training_dl: Training dataloader; 
                x: Test image; (C, W, H)
                y: Test label; 
                
            Returns:
                outputs: Return output list for given test images with dictionary structure as below 
                    outputs = [
                        {
                            'x': (first test image)
                            'y': (label for first test image) (not neccesary)
                            'p': (prediction for image)
                            '(conv_idx)': {
                                'test_r': relevance map for test image
                                'test_xy': {
                                    '(attr_idx)': (coordinates for current attribute with min_dist), 
                                    '(attr_idx)': (coordinates for current attribute with min_dist), 
                                    ...
                                }
                                'train_img': {
                                    '(attr_idx)': (train image similar to given min_dist_map of test image),
                                    '(attr_idx)': (train image similar to given min_dist_map of test image),
                                    ...
                                }
                                'train_xy': {
                                    '(attr_idx)': (coordinates of min_dist_map similar to min_dist_map of test image),
                                    '(attr_idx)': (coordinates of min_dist_map similar to min_dist_map of test image),
                                    ...
                                }
                            }
                        }
                    ]
        """
        outputs = []
        
        x = x.unsqueeze(0).to(device)
        if y!=None: y = y.unsqueeze(0).to(device) 

        r_attr, attr_xy, attr_dist_map, _, pred = self.get_mask(x=x,
                                                                target_idx=target_idx)
        nb_t_layers = len(r_attr)
        conv_idx_pointer = -1
        pred = pred[1]

        ###
        outputs_b = {}
        outputs_b['x'] = x.clone().detach().cpu()
        if y!=None: outputs_b['y'] = y.item()
        outputs_b['p'] = pred.item()
        ###
    
        for i in reversed(range(nb_t_layers)):
            cur_conv_idx = self.conv_idx[conv_idx_pointer]
            conv_idx_pointer -= 1
            if cur_conv_idx != target_idx: continue
            print("\tcur_conv_idx: ", cur_conv_idx)

            attr_xy_i = attr_xy[i].squeeze() # (nb_attrs, 2)
            attr_dist_map_i = attr_dist_map[i].squeeze()  # (nb_attrs, p-a+1, p-a+1)

            _, topk_idx = torch.topk(input=attr_dist_map_i.mean(dim=[1,2]), 
                                     k=attr_dist_map_i.mean(dim=[1,2]).shape[0], 
                                     dim=0)
            topk_idx = torch.flip(topk_idx, dims=[0]).tolist()

            attr_xy_dict, attr_dist_map_dict = get_xy_and_val_without_duplication(topk_idx=topk_idx,
                                                                                  xy=attr_xy_i,
                                                                                  d_map=attr_dist_map_i)
    
            min_l2d_img, min_l2d_xy = self._find_nearest_patches_in_trainset(trainset_dl=trainset_dl,
                                                                             dist_map_dict=attr_dist_map_dict,
                                                                             target_idx=cur_conv_idx, 
                                                                             test_pred=pred.item())

            ###
            outputs_i_info = {}
            outputs_i_info['test_r'] = r_attr.pop(i).clone().detach().cpu()
            outputs_i_info['test_xy'] = attr_xy_dict
            outputs_i_info['train_img'] = min_l2d_img
            outputs_i_info['train_xy'] = min_l2d_xy
            outputs_b[cur_conv_idx] = outputs_i_info
            ###
            
            gc.collect()
            torch.cuda.empty_cache()

        outputs.append(outputs_b)

        return outputs
    
    def _get_target_activaiton_and_pred(self, x, target_conv_idx):
        """
            Get target activation and prediction of the model for given input image
            
            Args:
                x: Input image batch; (B, C, W, H)
                target_conv_idx: Target conv layer to generate the activation for
            
            Returns:
            
        """
        
        with torch.no_grad():
            for key in self.layer_info.keys():
                for i, layer in enumerate(self.layer_info[key]):
                    x = layer.forward(x)
                    if i == target_conv_idx and key=='features':
                        target_a = x.clone().detach()

        # [0]: max values, [1]: index
        post_logits = torch.softmax(x, dim=-1)
        post_logits = torch.max(post_logits, 1)
        
        return target_a, post_logits
    
    def _find_nearest_patches_in_trainset(self, trainset_dl, dist_map_dict, target_idx, test_pred):
        """
            Find nearest patches from all train images for given test dist_map by attributes 
            
            Args: 
                trainset_dl: Dataloader of trainset
                dist_map_dict: Distance maps for the attributes of test image (no overlapping coordinates)
                target_idx: Index of target conv layer 
                test_pred: Prediction for test image; [0]: prediction values, [1]: predicted classes 
                
            Returns:
                
        """
        
        min_l2d = {k:sum(dist_map_dict.values()).mean()**3 for k in dist_map_dict.keys()}
        min_l2d_img = {}
        min_l2d_xy = {}
        
        for step, (x_batch, y_batch) in enumerate(trainset_dl): 
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            
            target_a, train_pred = self._get_target_activaiton_and_pred(x=x_batch,
                                                                        target_conv_idx=target_idx+1)
            
            train_valid_idx = get_valid_labels(self, y=y_batch, p=train_pred)
            x_batch = x_batch[train_valid_idx]
            y_batch = y_batch[train_valid_idx]
            target_a = target_a[train_valid_idx]
            train_vals = train_pred[0][train_valid_idx]
            train_labels = train_pred[1][train_valid_idx]
            
            # (B, nb_attrs, nb_sub_features, p-a+1, p-a+1)
            with torch.no_grad():
                all_dist_maps = self.cla_layers[target_idx]._get_all_distances_for_subfeatures(a=target_a, 
                                                                                               p=train_labels, 
                                                                                               y=None,
                                                                                               icp=self._get_icp_as_ts(self.sz_patches[target_idx]))
            
                all_dist_maps = all_dist_maps.detach().cpu()
                
                start_min = time.time()
                # number of subfeatures = dw*dh
                dw = target_a.shape[2] - self.sz_patches[target_idx] + 1
                dh = target_a.shape[3] - self.sz_patches[target_idx] + 1
                map_w, map_h = next(iter(dist_map_dict.values())).shape  # (p-a+1, p-a+1)
                for k in dist_map_dict.keys():
                    test_dist_map = dist_map_dict[k] 
                    test_dist_map = test_dist_map.reshape([1, map_w*map_h])

                    for b, dits_maps_b in enumerate(all_dist_maps):
                        if test_pred !=  y_batch[b].item(): continue # consider same class between train and test prediction
                        dist_maps_attr = dits_maps_b[k, :] # consider same attribute index between train and test 

                        for i, d_map_sf in enumerate(dist_maps_attr): 
                            old_min_l2d = min_l2d[k]
                            new_min_l2d = torch.cdist(test_dist_map, 
                                                      d_map_sf.reshape([1, map_w*map_h]).type(torch.DoubleTensor),
                                                      p=2)
                            
                            # get minimum l2-distance between train and test dist_map 
                            if new_min_l2d < old_min_l2d:
                                min_l2d[k] = new_min_l2d
                                min_l2d_img[k] = x_batch[b].clone().detach().cpu()
                                min_l2d_xy[k] = [i//dw, i%dh]
                                
                                print('\tk, b, i, min_l2d[k], min_l2d_xy[k] : ', k, b, i, min_l2d[k], min_l2d_xy[k] )
                                
                              
            gc.collect()
            torch.cuda.empty_cache()
            
        return min_l2d_img, min_l2d_xy
    
    def _print_output_epoch(self, e, val_dict):
        it_output = '\n---------------------------------\nFinish epoch: {} \n'.format(e)
        total = 0.
        for k in val_dict.keys():
            dp_mean = val_dict[k] / steps_of_train
            total += dp_mean
            it_output += '\tkey: {}, mean: {:.4f}\n'.format(k, dp_mean)

        it_output += '\tTotal mean: {:.4f}\n'.format(total/len(val_dict.keys()))
        it_output += '\n---------------------------------\n'
        
        return it_output 

In [12]:
class CLA_Layer(nn.Module):
    def __init__(self,
                 device,
                 idx,
                 layer_shape, 
                 nb_classes, 
                 nb_attrs,
                 sz_patch, 
                 sz_attr, 
                 lr,
                 max_epoches,
                 alpha, 
                 beta):
        
        super().__init__()
        self.device = device
        self.idx = idx
        self.feature_shape = layer_shape
        self.nb_classes = nb_classes
        self.nb_attrs = nb_attrs
        self.sz_patch = sz_patch
        self.sz_attr = sz_attr
        self.nb_attrs = nb_attrs
        self.max_epoches = max_epoches
        self.alpha = alpha
        self.beta = beta
        
        self.attr_shape, self.attr_vec, self.ones = self._init_attr_vector() 
        self.add_on_layers = self._init_add_on_layers()
       
        self._initialize_weights()
        
        self.opt = torch.optim.Adam(self.parameters(), lr=lr)
        self.scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer=self.opt,
                                                           lr_lambda=lambda epoch: 0.95 ** epoch,
                                                           last_epoch=-1,
                                                           verbose=False)
        
    def _init_attr_vector(self):
        f_dim = self.feature_shape[1]+1 # +1: for icp channel
            
        attr_shape = (self.nb_attrs, f_dim, self.sz_attr, self.sz_attr)
        attr_vec = nn.Parameter(torch.rand(attr_shape).cuda(), requires_grad=True)
        one = nn.Parameter(torch.ones(attr_shape).cuda(), requires_grad=False)
        
        return attr_shape, attr_vec, one
        
    def _init_add_on_layers(self):
        f_channel = self.attr_shape[1]
        
        add_on_layers = nn.Sequential(
            nn.Conv2d(in_channels=f_channel, out_channels=f_channel, kernel_size=1),     
            nn.SiLU(),
            nn.Conv2d(in_channels=f_channel, out_channels=f_channel, kernel_size=1),
            nn.SiLU()
        )   
        
        # if self.sz_patch > 11:
        #     add_on_layers.add_module('4', nn.MaxPool2d(kernel_size=2))
        #     add_on_layers.add_module('5', nn.Conv2d(in_channels=f_channel, out_channels=f_channel, kernel_size=1))
        #     add_on_layers.add_module('6', nn.SiLU())
        #     add_on_layers.add_module('7', nn.Conv2d(in_channels=f_channel, out_channels=f_channel, kernel_size=1))
        #     add_on_layers.add_module('8', nn.SiLU())
        
        add_on_layers.to(self.device)
        
        return add_on_layers
    
    def _initialize_weights(self):
        for m in self.add_on_layers.modules():
            if isinstance(m, nn.Conv2d):
                # every init technique has an underscore _ in the name
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                # nn.init.xavier_normal_(m.weight)

                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
    
    def _get_attribute_distance(self, x):
        '''
            Apply self.attribute_vectors as l2-convolution filters on input x
            
            Args:
                x: input feature generated by add_on_layers
            Returns:
                distances: distance between x and each attribute vectors 
        '''
        # x2: (b, c, w, h) -> x의 l2
        x2 = x ** 2 
        
        # ones: (nb_classes*nb_attr, c, w, h)
        # x2_patch_sum: (b, nb_classes*nb_attr, w-sz_attr+1, h-sz_attr+1)
        x2_patch_sum = F.conv2d(input=x2, weight=self.ones)

        # a2: (nb_classes*nb_attr, c, sz_attr, sz_attr)
        a2 = self.attr_vec ** 2
        
        # a2: (nb_classes*nb_attr) -> 각 attr별로 제곱 합 (l2)
        a2 = torch.sum(a2, dim=(1, 2, 3))

        # a2_reshape: (nb_classes*nb_attr, 1, 1)
        a2_reshape = a2.view(-1, 1, 1)

        # xa: (b, nb_classes*nb_attr, w-sz_attr+1, h-sz_attr+1)
        xa = F.conv2d(input=x, weight=self.attr_vec)
        
        # intermediate_result: (b, nb_classes*nb_attr, w-sz_attr+1, h-sz_attr+1)
        intermediate_result = - 2 * xa + a2_reshape  # use broadcast

        # distances: (b, nb_attrs, w-sz_attr+1, h-sz_attr+1)
        distances = F.relu(x2_patch_sum + intermediate_result)
        
        return distances
    
    def _get_all_distances_for_subfeatures(self, a, p, y, icp):
        if y == None: icp = icp[p, :]
        else: icp = icp[y, :]
        
        # each sub_feature: (B, C, sz_patch, sz_patch) and add icp to last channel of each sub feature 
        # then, get distance with each sub_feature 
        # distances: (B, nb_attrs, nb_sub_features, p-a+1, p-a+1)
        distances = torch.stack([self.forward(torch.cat((a[:, :, w:w+self.sz_patch, h:h+self.sz_patch].clone().detach(), 
                                                    icp), dim=1))
                                 for w in range(a.shape[2]-self.sz_patch+1)
                                 for h in range(a.shape[3]-self.sz_patch+1)], 
                                dim=2)
        
#         B, C, W, H = a.shape
#         nb_sf = (W-self.sz_patch+1) * (H-self.sz_patch+1)
#         # (b*nb_sf, nb_attrs, p-a+1, p-a+1)
#         distances = self.forward(torch.cat([(torch.cat((a[:, :, w:w+self.sz_patch, h:h+self.sz_patch], 
#                                                      icp), dim=1))
#                                          for w in range(W-self.sz_patch+1)
#                                          for h in range(H-self.sz_patch+1)], 
#                                         dim=0))
        
#         # distances = self.forward(sf_cat)  # (b*nb_sf, nb_attrs, p-a+1, p-a+1)
#         distances = distances.reshape([nb_sf, B, distances.shape[1], 
#                                          distances.shape[2], distances.shape[3]]).permute(1, 2, 0, 3, 4)
        
        return distances
    
    
    def _get_loss(self, y, p, d_pos_list, d_neg_list):
        valid_idx = get_valid_labels(y=y, p=p)
        
        # d_pos_list: list, len: nb_attrs, item shape: (b, nb_attrs, p-a+1, p-a+1)
        d_pos_diversity = torch.zeros_like(d_pos_list[0])
        d_neg_diversity = torch.zeros_like(d_neg_list[0])
        
        for i in range(self.nb_attrs):
            d_pos_diversity[:, i] = d_pos_list[i][:, i]
            d_neg_diversity[:, i] = d_neg_list[i][:, i]
        
        d_pos_diversity = d_pos_diversity.mean(dim=[2,3]).div(self.sz_attr).amax(dim=1)[valid_idx]
        d_neg_diversity = d_neg_diversity.mean(dim=[2,3]).div(self.sz_attr).mean(dim=1)[valid_idx]
        
        Delta = (self.beta*d_pos_diversity) - ((1-self.beta)*d_neg_diversity) 
        loss = F.softplus(self.alpha * Delta).mean()
        
        return loss, d_pos_diversity.mean(), d_neg_diversity.mean()

    def forward(self, x):
        """
            Propagate CLA_layer with input x
            
            Args: 
                x: Input, f_pos or f_neg (B, C, sz_patch, sz_patch)
            
            Return 
                distances: Distance map for each attribute on x 
                           (B, nb_attrs, sz_patch-sz_attr+1, sz_patch-sz_attr+1)
        """
        x = x / (x.norm(2, 1, keepdim=True) + 1e-4) 
        x = self.add_on_layers(x)
        
        return self._get_attribute_distance(x)
    
    def train(self, y, p, f_pos_list, f_neg_list):
        dist_pos_list = [self.forward(f_pos) for f_pos in f_pos_list]
        dist_neg_list = [self.forward(f_neg) for f_neg in f_neg_list]
        
        loss, pos_mean, neg_mean = self._get_loss(y, p, dist_pos_list, dist_neg_list)
        
        self.opt.zero_grad()
        loss.backward()
        self.opt.step()
        
        return loss, pos_mean, neg_mean   
    
    def predict(self, a, p, y, icp):
        """
            Predict minimum x and y coordinate from each attribute 
            
            Args:
                a: Target activation; (B, C, W, H)
                icp: Icp for patch size of current attribute 
                p: Prediction
                y: Label, default is None, if y is not None, y will be used in place of prediction 
                
            Returns:
                min_dist_xy: Coordinate of x and y of each attribute for given activation; (B, nb_attrs, 2)
        """
        B, C, W, H = a.shape
        
        distances = self._get_all_distances_for_subfeatures(a=a, p=p, y=y, icp=icp)
        
        # (B, nb_attrs, 2)
        attr_dist_xy = np.zeros([B, self.nb_attrs, 2], dtype=int)
        attr_dist_map = torch.zeros([B, self.nb_attrs, distances.shape[-1], distances.shape[-1]], dtype=float)
        dw = W-self.sz_patch+1
        dh = W-self.sz_patch+1
        for b, dist_b in enumerate(distances):
            # sub feature index with minimum distance for each attribute 
            attr_dist_xy_avg = [dist_attr.mean(dim=[1,2]).argmin(dim=0) for dist_attr in dist_b]
            # min_dist_val_by_attr = [dist_attr.mean(dim=[1,2]).amin(dim=0) for dist_attr in dist_b]
                
            for i in range(len(attr_dist_xy_avg)):
                attr_dist_xy[b, i, 0] =  attr_dist_xy_avg[i] // dw
                attr_dist_xy[b, i, 1] =  attr_dist_xy_avg[i] % dh
                
                attr_dist_map[b, i] = dist_b[i, attr_dist_xy_avg[i].item()].detach().cpu()
                
        return attr_dist_xy, attr_dist_map

In [13]:
clam = CLAM(model=model, 
            device=device, 
            icp_dict=icp_dict, 
            model_name=model_name,
            explainer_name=setting.NAME_OF_EXPLAINER_LRP,
            layer_sep_type=setting.LAYER_SEP_TYPE_CORE,
            nb_classes=200)

5
nb_attrs:  {3: 15, 8: 10, 15: 8, 22: 5, 29: 5}
sz_patches:  {3: 14, 8: 11, 15: 7, 22: 5, 29: 4}
sz_attrs:  {3: 6, 8: 5, 15: 3, 22: 2, 29: 1}
strength:  {3: 1.1, 8: 1.15, 15: 1.15, 22: 1.18, 29: 1.25}
alpha:  {3: 0.1, 8: 0.1, 15: 0.1, 22: 0.1, 29: 0.1}
beta:  {3: 0.5, 8: 0.6, 15: 0.6, 22: 0.6, 29: 0.6}
lr:  {3: 0.03, 8: 0.025, 15: 0.015, 22: 0.01, 29: 0.005}


In [14]:
indice = clam.conv_idx
for i in indice:
    path = '{}/ckpt_{}/{}.pth'.format(log_model_data_path,
                                      setting.NAME_OF_EXPLAINER_LRP,
                                      i)
    if os.path.exists(path):
        checkpoint = torch.load(path)
        clam.cla_layers[i].load_state_dict(checkpoint['layer_state_dict'])
        clam.cla_layers[i].opt.load_state_dict(checkpoint['optstate_dict'])
        print(i, 'last epoch:' , checkpoint['epoch'], 'dist_pos_mean: ', checkpoint['dist_pos'])

29 last epoch: 9 dist_pos_mean:  -331.2265388892059


In [15]:
print(clam.conv_idx)
target = 1
print('target: ', target)

[3, 8, 15, 22, 29]
target:  1


In [16]:
# clam.train(start_attr_idx=clam.conv_idx.index(3), target_idx=target)
clam.train(target_idx=target)

keys:  [3, 8, 15, 22, 29]
target_idx:  1
train_log_path:  logs/vgg16/imagenet/ckpt_LRP/log.txt
idx:  29
step: 160 in epoch 0 (idx: 29)
lr: 0.0031512470486230455
	loss 0.05221525579690933 
	pos_mean 32.9792 (amass: -365.8833) 
	neg_mean 373.0171

step: 320 in epoch 0 (idx: 29)
lr: 0.0031512470486230455
	loss 0.08775358647108078 
	pos_mean 51.1269 (amass: -370.5230) 
	neg_mean 398.3944

step: 480 in epoch 0 (idx: 29)
lr: 0.0031512470486230455
	loss 0.050270356237888336 
	pos_mean 38.2680 (amass: -370.7122) 
	neg_mean 456.8524

step: 640 in epoch 0 (idx: 29)
lr: 0.0031512470486230455
	loss 0.09503360092639923 
	pos_mean 31.5670 (amass: -376.1341) 
	neg_mean 436.9269

step: 800 in epoch 0 (idx: 29)
lr: 0.0031512470486230455
	loss 0.1042313277721405 
	pos_mean 33.8829 (amass: -378.8345) 
	neg_mean 393.7937

step: 960 in epoch 0 (idx: 29)
lr: 0.0031512470486230455
	loss 0.04775083437561989 
	pos_mean 38.5342 (amass: -377.7707) 
	neg_mean 420.2922

step: 1120 in epoch 0 (idx: 29)
lr: 0.003151

KeyboardInterrupt: 

In [ ]:
for x_batch, y_batch in val_dl:
    r_attr, attr_xy, _, _, pred = clam.get_mask(x=x_batch, target_idx=target)
    r_orig = clam.get_origal_rs(x=x_batch, target_idx=target) 
    
    break

idx_list = clam.conv_idx.copy()
# idx_list.insert(0, -1)
attr_masks = []
for i, idx in enumerate(reversed(idx_list)):
    if target <= idx:
        rev_i = len(attr_xy)-i-1
        attr_masks.insert(0, clam._get_attribute_mask(xy=attr_xy[rev_i],
                                                   r_shape=r_attr[rev_i].shape,
                                                   idx=idx))

In [ ]:
nb_batch = r_attr[0].shape[0]
col = len(r_attr)+1
nb_list = 3
row = nb_batch*nb_list
uf = 3

fig, axs = plt.subplots(row, col, figsize=(col*uf, row*uf))

for b in range(row):
    for i in range(col):
        x = int(i/col) + b
        y = int(i%col) 
        
        # axs[x,y].set_xticks([])
        # axs[x,y].set_yticks([])
        
        if i == 0:
            cur_row = b//nb_list
            if y_batch[cur_row] == pred[1][cur_row]:
                img = x_batch[cur_row].permute(1,2,0).squeeze().detach().cpu()
            else: 
                if b % nb_list == 0 or b % nb_list == 2:
                    img = x_batch[cur_row].permute(1,2,0).squeeze().detach().cpu()
                else:
                    img = torch.zeros(x_batch[b//nb_list].shape).permute(1,2,0)
                
            axs[x,y].imshow(img)
            
        else: 
            if b%nb_list == 0:
                hm = r_orig[i-1][b//nb_list].permute(1,2,0).max(dim=-1)[0].detach().cpu()
            elif b%nb_list == 1:
                hm = attr_masks[i-1][b//nb_list].permute(1,2,0).max(dim=-1)[0].detach().cpu()
            else:
                hm = r_attr[i-1][b//nb_list].permute(1,2,0).max(dim=-1)[0].detach().cpu()
            
            axs[x,y].imshow(hm, vmin=-hm.max(), vmax=hm.max(), cmap='seismic')


<!-- ## Backup

push 하는 부분 activation 쓰는거

# Class-wise Layer-wise Attribute Model 
class CLAM(nn.Module):
    def __init__(self, 
                 model, 
                 device, 
                 icp_dict,
                 model_name: str=None, 
                 explainer_name: str=setting.NAME_OF_EXPLAINER_LRP,
                 layer_sep_type: str='FULL', 
                 nb_classes: int=0):
        super().__init__()
        
        self.model = model
        self.model.eval()
        self.device = device
        self.icp_dict = icp_dict
        self.model_name = model_name
        self.explainer_name = explainer_name
        self.layer_sep_type = layer_sep_type
        self.nb_classes = nb_classes
        
        self.model_info = model_info
        self.layer_info = self.model_info['layer']
        self.shape_info = self.model_info['shape']
        self.layer_names = self.model_info['name']
        
        self.explainer = self._init_explainer(exp_name=self.explainer_name)
        
        self.conv_idx, self.cla_layers, self.nb_attrs, self.sz_patches, \
            self.sz_attrs, self.lr, self.max_epoches, self.strength = self._init_cla_layers()
        # self.icp = self._icp_dict_to_ts(icp_dict)
        # self.nb_epochs = sum(self.max_epoches.values())
        self.nb_epochs = self.max_epoches * len(self.cla_layers.keys())
        
    def _init_explainer(self, exp_name):
        if exp_name == setting.NAME_OF_EXPLAINER_LRP:
            explainer = lrp.LRP(layer_info=self.layer_info)
            
        elif exp_name == setting.NAME_OF_EXPLAINER_CLRP:
            explainer = clrp.CLRP(layer_info=self.layer_info)
            
        elif exp_name == setting.NAME_OF_EXPLAINER_SGLRP:
            explainer = sglrp.SGLRP(layer_info=self.layer_info)
            
        else:
            print("Invalid explainer name, exiting...")
            exit()
            
        return explainer
    
    def _init_params(self, conv_idx, conv_shape):
        
        nb_attrs = {}
        sz_patches = {}
        sz_attrs = {}
        strength = {}
        alpha = {}
        beta = {}
        lr = {}
        max_epoches = setting.max_epoches
        
        for i, (idx, shape) in enumerate(zip(conv_idx, conv_shape)):
            if self.layer_sep_type == setting.LAYER_SEP_TYPE_FULL:
                # nb_attrs
                if setting.nb_attrs == 'MANUAL': 
                    if model_name == 'vgg16':
                        nb_attrs[idx] = setting.nb_attrs_vgg16_full[i]
                    
                # sz_patches
                if setting.sz_patches == 'MANUAL': 
                    if model_name == 'vgg16' and dataset_name == 'cub':
                        sz_patches[idx] = setting.sz_patches_vgg_cub_full[i]
                else:
                    sz_patches[idx] = int(shape[-1]/nb_attrs[idx])
                
                # sz_attrs
                if setting.sz_attrs == 'MANUAL': 
                    if model_name == 'vgg16' and dataset_name == 'cub':
                        sz_attrs[idx] = setting.sz_attrs_vgg16_cub_full[i]
                else:
                    sz_attrs[idx] = int((sz_patches[idx]-1)/2)
                
                # strength
                if setting.strength == 'MANUAL': 
                    if model_name == 'vgg16':
                        strength[idx] = setting.strength_vgg16_full[i]
                        
                # alpah 
                if setting.alpha == 'MANUAL': 
                    if model_name == 'vgg16':
                        alpha[idx] = setting.alpha_vgg16_full[i]
                        
                # beta
                if setting.beta == 'MANUAL': 
                    if model_name == 'vgg16':
                        beta[idx] = setting.beta_vgg16_full[i]
                
                # lr
                if setting.lr == 'MANUAL': 
                    if model_name == 'vgg16':
                        lr[idx] = setting.lr_vgg16_full[i]
                else:
                    lr[idx] =  0.01 * (sz_attrs[idx]/2)
                    
        print('nb_attrs: ', nb_attrs)
        print('sz_patches: ', sz_patches)
        print('sz_attrs: ', sz_attrs)
        print('strength: ', strength)
        print('alpha: ', alpha)
        print('beta: ', beta)
        print('lr: ', lr)
            
        return nb_attrs, sz_patches, sz_attrs, max_epoches, strength, alpha, beta, lr
           
    
    def _init_cla_layers(self):
            
        cla_layers = {}
        feature_key = list(self.shape_info.keys())[0]
        
        if self.layer_sep_type == setting.LAYER_SEP_TYPE_FULL:
            conv_idx = [idx for idx, name in enumerate(self.layer_names[feature_key]) if 'CONV' in name]
            
        conv_shapes = list(map(self.shape_info['features'].__getitem__, conv_idx))
        
        nb_attrs, sz_patches, sz_attrs, max_epoches, \
            strength, alpha, beta, lr = self._init_params(conv_idx, conv_shapes)
        
        for idx in conv_idx:
            cla_layers[idx] = CLA_Layer(device=self.device, 
                                        idx=idx+1, 
                                        layer_shape=self.shape_info[feature_key][idx], 
                                        nb_classes=self.nb_classes, 
                                        nb_attrs=nb_attrs[idx],
                                        sz_patch=sz_patches[idx], 
                                        sz_attr=sz_attrs[idx], 
                                        max_epoches=max_epoches, 
                                        alpha=alpha[idx], 
                                        beta=beta[idx], 
                                        lr=lr[idx])

        return conv_idx, cla_layers, nb_attrs, sz_patches, sz_attrs, lr, max_epoches, strength
    
    
    def _get_icp_as_ts(self, sz_p): 
        # (nb_classes, 1, sz_p, sz_p)
        return torch.tensor(self.icp_dict[sz_p])[:, None, :, :].cuda()
    
    def _get_rand_xy_exclusive_of_t_list(self, t_list, max_val):
        rand = np.zeros(t_list.shape, dtype=int)
        
        for k, l in enumerate(t_list): 
            nb_list_items = len(l)
            while(True):
                rand_xy = [random.randint(0, max_val) for i in range(nb_list_items)]
                checker = [rand_xy[j] == l[j]  for j in range(nb_list_items)]
                if True not in checker: 
                    rand[k] = rand_xy
                    break 
                    
        return rand
    
    def _get_r_max_idx(self, r, idx):
        r_2d = r.max(dim=1)[0]
        r_2d = F.avg_pool2d(input=r_2d, 
                            kernel_size=(self.sz_patches[idx], 
                                         self.sz_patches[idx]), 
                            stride=1)
        
        b, rw, rh = r_2d.shape
        k = self.nb_attrs[idx]
        r_1d = r_2d.reshape([b, rw*rh]) #.clone().detach()
        
        r_topk_val, r_topk_idx = torch.topk(r_1d, k=k, dim=1)
        
        # r_topk_idx -> row, col 나눔
        r_topk_row = np.zeros(r_topk_idx.shape).astype(np.int32)
        r_topk_col = np.zeros(r_topk_idx.shape).astype(np.int32)
        for i, idx in enumerate(r_topk_idx):
            r_topk_row[i] = idx.cpu().numpy()//rw
            r_topk_col[i] = idx.cpu().numpy()%rw
        
        return r_topk_row, r_topk_col
    
    def _generate_pos_neg(self, r, a, y, idx):
        """
            Generate pos/neg inputs for training cla_layers
            
            Args:
                r: Relevance (B, C, W, H)
                a: Activation (B, C, W, H)
                y: Label (B, nb_classes)
                idx: Current conv idx
                
           Returns: 
               f_pos: Pos sub feature (nb_attrs, B, C, sz_patch, sz_patch)
               f_neg: Neg sub feature (nb_attrs, B, C, sz_patch, sz_patch)
               r_topk_row: Top-k row index of relevance (k == nb_attr)
               r_topk_col: Top-k col index of relevance (k == nb_attr)
               
        """
        r_topk_row, r_topk_col = self._get_r_max_idx(r, idx)
        
        r2d_max_idx = r.shape[-1] - self.sz_patches[idx]
        r_row_rand_idx = self._get_rand_xy_exclusive_of_t_list(t_list=r_topk_row, 
                                                               max_val=r2d_max_idx)
        r_col_rand_idx = self._get_rand_xy_exclusive_of_t_list(t_list=r_topk_col,
                                                               max_val=r2d_max_idx)
        
        # (nb_attr, b, d, p, p)
        f_pos = torch.zeros([self.nb_attrs[idx], a.shape[0], a.shape[1], 
                             self.sz_patches[idx], self.sz_patches[idx]]).to(device)
        f_neg = torch.zeros([self.nb_attrs[idx], a.shape[0], a.shape[1], 
                             self.sz_patches[idx], self.sz_patches[idx]]).to(device) 
        
        r_topk_row = r_topk_row.T
        r_topk_col = r_topk_col.T
        r_row_rand_idx = r_row_rand_idx.T
        r_col_rand_idx = r_col_rand_idx.T
        
        
        # Extract sub_feature using top-k index for pos and rand index for neg
        for i in range(self.nb_attrs[idx]):
            indices = []
            for j, a_b in enumerate(a):
                f_pos[i,j] = a_b[:, r_topk_row[i][j]:r_topk_row[i][j]+self.sz_patches[idx],
                             r_topk_col[i][j]:r_topk_col[i][j]+self.sz_patches[idx]]
                f_neg[i,j] = a_b[:, r_row_rand_idx[i][j]:r_row_rand_idx[i][j]+self.sz_patches[idx],
                             r_col_rand_idx[i][j]:r_col_rand_idx[i][j]+self.sz_patches[idx]]
        
        # Add icp to last channel in pos/neg
        icp = torch.stack([self._get_icp_as_ts(self.sz_patches[idx]) 
                           for i in range(self.nb_attrs[idx])], dim=0)
        
        f_pos = torch.cat((f_pos, icp[:, y, :]), dim=2)
        
        rand_p = self._get_rand_xy_exclusive_of_t_list(t_list=y.unsqueeze(1), 
                                                       max_val=self.nb_classes-1)
        rand_p = torch.from_numpy(rand_p).squeeze()
        f_neg = torch.cat((f_neg, icp[:, rand_p, :]), dim=2)
        
        return f_pos, f_neg, r_topk_row, r_topk_col
    
    def _get_mask_for_strength(self, idx):
        s = self.strength[idx]
        p = self.sz_patches[idx] 
        mask = np.zeros([p*2-1, p*2-1])
        w, h = mask.shape
        cp_x, cp_y = w//2, h//2
        
        for i in range(w):
            for j in range(h):        
                dis_x = np.abs(cp_x-i)
                dis_y = np.abs(cp_y-j)
                max_dis = dis_x if dis_x-dis_y>0 else dis_y

                if dis_x==0 and dis_y==0:
                    mask[i, j] = s
                else:
                    for k in range(1, p):
                        if max_dis==k:
                            str_v = s*(0.95**k)
                            if str_v < 1.0:
                                str_v = 1.0
                            mask[i, j] = str_v
                            break
        
        return torch.from_numpy(mask).to(device)
    
    def _get_new_pos(self, base_pos, w, h, idx, mask):
        m_w, m_h = mask.shape
        
        if base_pos-self.sz_patches[idx]+1 < 0: 
            s_pt = 0
            m_s_pt = -(base_pos-self.sz_patches[idx]+1)
        else:
            s_pt = base_pos-self.sz_patches[idx]+1
            m_s_pt = 0
        
        if base_pos+self.sz_patches[idx] > w:
            e_pt = w
            m_e_pt = w - base_pos + (m_w-self.sz_patches[idx])
        else: 
            e_pt = base_pos+self.sz_patches[idx]
            m_e_pt = m_w

        return s_pt, e_pt, m_s_pt, m_e_pt
    
    def _apply_min_dist_to_r(self, r, xy, idx, r_idx=None):
        mask = self._get_mask_for_strength(idx) # (p-a+1, p-a+1)
        
        _, _, w, h = r.shape # r: (B, C, W, H)
        for i, r_b in enumerate(r):
            if r_idx != None:
                r_idx_topk = r_idx[i]  # k==nb_attrs[idx]
            else:
                r_idx_topk = [[0, 0] for i in range(self.nb_attrs[idx])]
                
            for j in range(xy.shape[1]):
                r_x, r_y = r_idx_topk[j]
                d_x, d_y = xy[i, j]
                
                x_start, x_end, mx_s_pt, mx_e_pt = self._get_new_pos(r_x+d_x, w, h, idx, mask.clone().detach())
                y_start, y_end, my_s_pt, my_e_pt = self._get_new_pos(r_y+d_y, w, h, idx, mask.clone().detach())

                r_b[:, x_start:x_end, y_start:y_end] *= mask[mx_s_pt:mx_e_pt, my_s_pt:my_e_pt]
                        
        return r
    
    
    def _combine_r_and_dist(self, r, dist_list, idx, r_topk_row, r_topk_col):
        """
            Generate new relevance map with dist_list 
        
        """
        sz_batch = r.shape[0]
        min_dist_xy = np.zeros([sz_batch, self.nb_attrs[idx], 2], dtype=int) # 2: row, col val
        for i, dist in enumerate(dist_list):
            b, a, w, h = dist.shape  # (b, nb_attrs, p-a+1, p-a+1)
            dist = dist.reshape(b, a, w*h)
            min_dist_pos = dist.argmin(dim=-1)
            min_dist_pos = min_dist_pos[:, i]
            for j, pos in enumerate(min_dist_pos):
                min_dist_xy[j, i, 0] = pos//w
                min_dist_xy[j, i, 1] = pos%h
        
        r_topk_row = r_topk_row.T
        r_topk_col = r_topk_col.T
        
        r_idx = []
        for i in range(sz_batch):
            r_idx_b = [[r_topk_row[i, j], r_topk_col[i, j]] for j in range(self.nb_attrs[idx])]
            r_idx.append(r_idx_b)
         
        return self._apply_min_dist_to_r(r, min_dist_xy, idx, r_idx)
            
    def _get_pre_r_and_a(self, x, y, target_idx):
        r = None
        dist_pos_list = None
        rest_a = None
        pred = None
        
        s_idx = -1
        t_idx = 0
        
        for idx in reversed(self.conv_idx):
            if r == None: r = x
            else: s_idx = t_idx
            t_idx = self.cla_layers[idx].idx
            
            r, a, p, rest_a = self.explainer.forward(r=r, 
                                                     y=y,
                                                     acts=rest_a, 
                                                     s_idx=s_idx, 
                                                     t_idx=t_idx)
            if pred == None: pred = p
                
            if idx == target_idx: break
            else:
                with torch.no_grad():
                    f_pos, _, r_topk_row, r_topk_col = self._generate_pos_neg(r, a, y, idx)
                    dist_pos_list = [self.cla_layers[idx](f_p) for f_p in f_pos]
                    r = self._combine_r_and_dist(r, dist_pos_list, idx, r_topk_row, r_topk_col)
                    
        return r, a, pred
    
    def _get_attribute_mask(self, attr_xy, r_shape, idx):
        """
            Generate a mask for min_dist point used for relevance refinement  
        """
        b, _, w, h = r_shape
        
        m = torch.ones([b, 1, w, h]).to(device)
        if idx in self.conv_idx:
            m = self._apply_min_dist_to_r(m, attr_xy, idx)
            m -= 1.0
        
        return m.clone().detach().cpu()
    
    def get_origal_rs(self, x, y=None, target_idx=-1):
        x = x.to(self.device)
        if y is not None: y = y.to(self.device)
        
        r = None
        rest_a = None
        rs = []
        
        s_idx = -1
        t_idx = 0
        
        # for propagating to input layer, add idx -1
        
        conv_idx_expand = self.conv_idx.copy()  
        if target_idx == -1:
            conv_idx_expand.insert(0, -1) # for input layer 
        
        for idx in reversed(conv_idx_expand):
            if r == None: r = x
            else: s_idx = t_idx
            
            if idx == -1: t_idx = 0
            else: t_idx = self.cla_layers[idx].idx
            
            r, a, _, rest_a = self.explainer.forward(r=r, 
                                                     y=y,
                                                     acts=rest_a, 
                                                     s_idx=s_idx, 
                                                     t_idx=t_idx)
            rs.insert(0, r.clone().detach().cpu())
            
            if idx == target_idx: break
            
        return rs
                
    def train(self, start_attr_idx=-1, target_idx=0):
        keys = self.conv_idx
        print('keys: ', keys)
        
        # === for test
        if start_attr_idx > 0:
            keys = keys[:start_attr_idx+1]
        print('target_idx: ', target_idx)
        
        train_log_path = os.path.join(log_model_data_path, 'ckpt_{}/log.txt'.format(self.explainer_name))
        print('train_log_path: ', train_log_path)
        with open(train_log_path, 'w') as f:
            f.write('')
        
        dist_pos_neg_diff = {k: 0. for k in keys}
        max_stopping_cnt = 3 
        
        for i, idx in enumerate(reversed(keys)):
            attr_d, attr_w, attr_h = self.cla_layers[idx].attr_shape[1:]
            min_diff_mean = attr_d * attr_w * attr_h * self.nb_attrs[idx]
            early_stopping_cnt = 0
            
            for epoch in range(self.max_epoches):
                
                dist_pos_neg_diff[idx] = 0.
                
                for step, (x_batch, y_batch) in enumerate(train_dl): 
                    x_batch = x_batch.to(device)
                    y_batch = y_batch.to(device)
                    
                    r, a, pred = self._get_pre_r_and_a(x=x_batch, 
                                                       y=y_batch,
                                                       target_idx=idx)
                    
                    f_pos, f_neg, _, _ = self._generate_pos_neg(r=r,
                                                                a=a,
                                                                y=y_batch, 
                                                                idx=idx)
                    
                    
                    loss, pos_mean, neg_mean = self.cla_layers[idx].train(y=y_batch, 
                                                                          p=pred, 
                                                                          f_pos_list=f_pos,
                                                                          f_neg_list=f_neg)
                    
                    dist_pos_neg_diff[idx] += pos_mean.item() - neg_mean.item()
                    
                    
                    if step % int(steps_of_train/2) == 0 and step > 0:
                        p_output = 'step: {} in epoch {} (idx: {})\n'.format(step, epoch, idx)
                        p_output += 'lr: {}\n'.format(self.cla_layers[idx].opt.param_groups[0]['lr'])
                        p_output += '\tloss {} \n\tpos_mean {:.4f} (amass: {:.4f}) \n\tneg_mean {:.4f}\n'.format(loss.item(), 
                                                                                                                 pos_mean.item(),
                                                                                                                 dist_pos_neg_diff[idx]/step,
                                                                                                                 neg_mean.item())
                        print(p_output)
                        with open(train_log_path, 'a') as f:
                            f.write(p_output)  
                    
                cur_idx_dist_diff_mean = dist_pos_neg_diff[idx] / steps_of_train
                if cur_idx_dist_diff_mean < min_diff_mean:
                    min_diff_mean = cur_idx_dist_diff_mean
                    early_stopping_cnt = 0

                    print('Save model!')
                    torch.save({'epoch': epoch,
                                'layer_state_dict': self.cla_layers[idx].state_dict(),
                                'optstate_dict': self.cla_layers[idx].opt.state_dict(),
                                'dist_pos': dist_pos_neg_diff[idx] / step
                               }, '{}/ckpt_{}/{}.pth'.format(log_model_data_path, 
                                                             self.explainer_name,
                                                             idx))
                    self.cla_layers[idx].scheduler.step()

                else: 
                    early_stopping_cnt += 1

                e_output = self._print_output_epoch(epoch, dist_pos_neg_diff)
                if early_stopping_cnt == max_stopping_cnt: 
                    e_output += 'Skip to next epoch \n'
                else:
                    e_output += 'Current stopping_cnt: {}\n\n'.format(early_stopping_cnt)
                print('e_output: ', e_output)
                with open(train_log_path, 'a') as f:
                    f.write(e_output)

                if early_stopping_cnt == max_stopping_cnt:
                    gc.collect()
                    torch.cuda.empty_cache()
                    break
                               
            if idx == target_idx: break
                
        print("FINISH")
        
    def get_mask(self, x, y=None, target_idx=-1):
        """
            Propagate CLA-layers on x batch
            
            Args: 
                x: Input image batch; (B, C, W, H)
                y: Label; (B, nb_classes), If label is None, prediction will be used as label 
                target_idx: Target conv idx for stopping propagation
            Returns:
                r: Generated relevance combining with mask predicted by attributes
                r_orig: Original relevance on input 
        """
        x = x.to(self.device)
        if y is not None: y = y.to(self.device)

        r = None
        rest_a = None
        pred = None
        r_attr = []
        acts = []
        attr_xy = []
        attr_map = []
        
        s_idx = -1
        t_idx = 0
        
        # for propagating to input layer, add idx -1
        conv_idx_expand = self.conv_idx.copy()
        if target_idx == -1:
            conv_idx_expand.insert(0, -1)
        
        for idx in reversed(conv_idx_expand):
            if r == None: r = x.clone().detach()
            else: s_idx = t_idx
            
            if idx == -1: t_idx = 0
            else: t_idx = self.cla_layers[idx].idx
            
            r, a, p, rest_a = self.explainer.forward(r=r, 
                                                     y=None,
                                                     acts=rest_a, 
                                                     s_idx=s_idx, 
                                                     t_idx=t_idx)
            if pred == None: pred = p
            
            if idx == -1: 
                r_attr.insert(0, r.clone().detach().cpu())
                attr_xy.insert(0, 0)
                attr_map.insert(0, 0)
                acts.insert(0, a.clone().detach().cpu())
                break
                
            else:            
                with torch.no_grad():
                    min_dist_xy, min_dist_map = self.cla_layers[idx].predict(a=a, 
                                                                             p=pred, 
                                                                             y=y, 
                                                                             icp=self._get_icp_as_ts(self.sz_patches[idx]))
                    r = self._apply_min_dist_to_r(r, min_dist_xy, idx)
                    r_attr.insert(0, r.clone().detach().cpu())
                    attr_xy.insert(0, min_dist_xy)
                    attr_map.insert(0, min_dist_map)
                    acts.insert(0, a.clone().detach().cpu())
            
            gc.collect()
            torch.cuda.empty_cache()

            if idx == target_idx: break
        
        return r_attr, attr_xy, attr_map, acts, pred
    
    def push_attributes(self, trainset_dl, testset, target_idx=0):
        """
            Push each attribute in each layer to the nearest patch in the training set
            
            Args:
                training_dl: Training dataloader; 
                testset: List including images and labels from dataiter; [0]: (B, C, W, H), [1]: (B,)
                
            Returns:
                
        
        """
        
        ####
        tot_cnt_p = 1e-6
        tot_cnt_n = 1e-6
        md_cor_p = 1e-6
        mc_cor_p = 1e-6
        ml2_cor_p = 1e-6
        md_cor_n = 1e-6
        mc_cor_n = 1e-6
        ml2_cor_n = 1e-6
        ###
        
        # for x, y in zip(*testset):
        dataiter = iter(val_dl)
        for i, testset in enumerate(dataiter):
            print('\n------------------------\nSTEP: ', i)
            
            for x, y in zip(*testset): 
                x = x.unsqueeze(0).to(device)
                y = y.unsqueeze(0).to(device) 

                print(x.shape)

                r_attr, attr_xy, attr_dist_map, acts, pred = self.get_mask(x,
                                                                      target_idx=target_idx)
                
                nb_t_layers = len(r_attr)
                conv_idx_pointer = -1
                for i in reversed(range(nb_t_layers)):
                    cur_conv_idx = self.conv_idx[conv_idx_pointer]

                    attr_xy = attr_xy[i].squeeze() # (nb_attrs, 2)
                    attr_dist_map = torch.from_numpy(attr_dist_map[i]).squeeze()  # (nb_attrs, p-a+1, p-a+1)
                    
                    dist_map_avg = attr_dist_map.mean(dim=[1,2])
                    _, topk_idx = dist_map_avg.topk(k=dist_map_avg.shape[0], dim=0)
                    topk_idx = torch.flip(topk_idx, dims=[0]).tolist()
                    
                    
                    
                    attr_xy_dict, attr_dist_map_dict = get_xy_and_val_without_duplication(topk_idx=topk_idx,
                                                                                          xy=attr_xy,
                                                                                          d_map=attr_dist_map)
                    
                    print('Y - Pred: ', y.item(), ' - ', pred[1].item())
                    
                    a = acts[i].squeeze()
                    sub_f_dict = {k:a[:, 
                                      min_xy_dict[k][0]:min_xy_dict[k][0]+self.sz_patches[cur_conv_idx], 
                                      min_xy_dict[k][1]:min_xy_dict[k][1]+self.sz_patches[cur_conv_idx]].detach().cpu()\
                                  for k in min_xy_dict.keys()}

                    min_diff_y, min_csim_y, min_l2_y = self._find_nearest_patches_in_trainset(trainset_dl=trainset_dl,
                                                           min_dist_dict=min_dist_dict, 
                                                           sub_feature_dict=sub_f_dict,
                                                           target_idx=cur_conv_idx)
                    #####
                    test_p = pred[1].item()
                    
                    if y.item()==test_p:
                        tot_cnt_p += len(list(min_diff_y.keys()))
                        for k in min_diff_y.keys():
                            md_y = min_diff_y[k]
                            mc_y = min_csim_y[k]
                            ml2_y = min_l2_y[k]

                            if test_p == md_y:
                                md_cor_p += 1
                            if test_p == mc_y:
                                mc_cor_p += 1
                            if test_p == ml2_y:
                                ml2_cor_p += 1
                        
                    else:
                        tot_cnt_n += len(list(min_diff_y.keys()))
                        for k in min_diff_y.keys():
                            md_y = min_diff_y[k]
                            mc_y = min_csim_y[k]
                            ml2_y = min_l2_y[k]

                            if test_p == md_y:
                                md_cor_n += 1
                            if test_p == mc_y:
                                mc_cor_n += 1
                            if test_p == ml2_y:
                                ml2_cor_n += 1
                        

                    ####

                    conv_idx_pointer -= 1

                ####
                tot_cnt = tot_cnt_p+tot_cnt_n
                md_cor = md_cor_p+md_cor_n
                mc_cor = mc_cor_p+mc_cor_n
                ml2_cor = ml2_cor_p+ml2_cor_n
                
                out = '\nTotal count: {} (p: {}, n: {})\n'.format(int(tot_cnt), int(tot_cnt_p), int(tot_cnt_n))
                out += '\t min_dist - correct num: {} ({:.2f})\n'.format(int(md_cor), (md_cor/tot_cnt*100))
                out += '\t\t min_dist p - correct num: {} ({:.2f})\n'.format(int(md_cor_p), (md_cor_p/tot_cnt_p*100))
                out += '\t\t min_dist n - correct num: {} ({:.2f})\n'.format(int(md_cor_n), (md_cor_n/tot_cnt_n*100))
                out += '\t max_csim - correct num: {} ({:.2f})\n'.format(int(mc_cor), (mc_cor/tot_cnt*100))
                out += '\t\t max_csim p - correct num: {} ({:.2f})\n'.format(int(mc_cor_p), (mc_cor_p/tot_cnt_p*100))
                out += '\t\t max_csim n - correct num: {} ({:.2f})\n'.format(int(mc_cor_n), (mc_cor_n/tot_cnt_n*100))
                out += '\t min_l2d - correct num: {} ({:.2f})\n'.format(int(ml2_cor), (ml2_cor/tot_cnt*100))
                out += '\t\t min_l2d p - correct num: {} ({:.2f})\n'.format(int(ml2_cor_p), (ml2_cor_p/tot_cnt_p*100))
                out += '\t\t min_l2d n - correct num: {} ({:.2f})\n'.format(int(ml2_cor_n), (ml2_cor_n/tot_cnt_n*100))
                
                print(out)
                pathh = '/archive/workspace/XAI/research/CLAM/test.txt'
                with open(pathh, 'a') as f:
                    f.write(out)
                ###
                

    
    def _find_nearest_patches_in_trainset(self, trainset_dl, min_dist_dict, sub_feature_dict, target_idx):
        """
            1. 우선 pred 관계 없이 전체다 해보고, pred와 같은 클래스를 잘 찾으면 best
            1-1. dist, csim, ㅣ2-dist 중 젤 괜찮은거로 (dist가 best)
            2. 못 찾으면 pred와 같은 클래스만 대상으로 해야될듯?
            2-1. pred과 같은 클래스의 training image 중에서, pred가 틀린건 제외
            
            ? 현재 attr 별로 학습되었는데, 여기서도 고려? -> 우선 한번 해봐야될듯
        
        """
        # initialize min_diff; key=attr_num, value: difference value between train and test min_dist
        min_diff = {k:min_dist_dict[k]**2 for k in min_dist_dict.keys()}
        min_diff_y = {}
        min_diff_attr_num = {}
        
        min_csim = {k:0 for k in min_dist_dict.keys()}
        min_csim_y = {}
        min_csim_attr_num = {}
        csim = nn.CosineSimilarity(dim=0, eps=1e-6)
        
        min_l2 = {k:min_dist_dict[k]**2 for k in min_dist_dict.keys()}
        min_l2_y = {}
        min_l2_attr_num = {}
        
        
        for step, (x_batch, y_batch) in enumerate(trainset_dl): 
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            
            # r_attr, attr_xy, attr_val, acts, pred
            _, attr_xy, attr_val, acts, train_p = self.get_mask(x=x_batch,
                                                                target_idx=target_idx)
            train_xy = attr_xy[0]
            train_min_dist = attr_val[0]
            train_a = acts[0]
            train_p = train_p[1]
            
            for k in min_dist_dict.keys():
                test_min_dist = min_dist_dict[k]
                
                for b, t_min_dist in enumerate(train_min_dist):
                    # 맞춘것만
                    if train_p[b].item() != y_batch[b].item(): continue
                    
                    for i, tmd in enumerate(t_min_dist):
                        new_diff = np.abs(test_min_dist-tmd)
                        old_diff = min_diff[k]
                        
                        if new_diff < old_diff:
                            min_diff[k] = new_diff
                            min_diff_y[k] = y_batch[b].item()
                            min_diff_attr_num[k] = i
                            
            for k in min_dist_dict.keys():
                test_sub_f = sub_feature_dict[k]
                d, p, p = test_sub_f.shape
                test_sub_f = test_sub_f.reshape(d*p*p)
                
                for b, t_xy in enumerate(train_xy):
                    if train_p[b].item() != y_batch[b].item(): continue
                    
                    for i, xy in enumerate(t_xy):
                        x, y = xy[0], xy[1]
                        train_sub_f = train_a[b, :, 
                                              x:x+self.sz_patches[target_idx],
                                              y:y+self.sz_patches[target_idx]].detach().cpu()
                        train_sub_f = train_sub_f.reshape(d*p*p)
                        
                        
                        new_csim = csim(test_sub_f, train_sub_f)
                        old_csim = min_csim[k]
                        
                        new_l2 = torch.cdist(test_sub_f.unsqueeze(0), train_sub_f.unsqueeze(0), p=2)
                        old_l2 = min_l2[k]
                        
                        if new_csim > old_csim:
                            min_csim[k] = new_csim
                            min_csim_y[k] = y_batch[b].item()
                            min_csim_attr_num[k] = i
                        
                        if new_l2 < old_l2:
                            min_l2[k] = new_l2
                            min_l2_y[k] = y_batch[b].item()
                            min_l2_attr_num[k] = i
                            
                            
            
        print('min_diff: ', min_diff)
        print('min_diff_y: ', min_diff_y)
        print('min_diff_attr_num: ', min_diff_attr_num, '\n')
        
        print('min_csim: ', min_csim)
        print('min_csim_y: ', min_csim_y)
        print('min_csim_attr_num: ', min_csim_attr_num, '\n')
        
        
        print('min_l2: ', min_l2)
        print('min_l2_y: ', min_l2_y)
        print('min_l2_attr_num: ', min_l2_attr_num)
            
        return min_diff_y, min_csim_y, min_l2_y
            
        
    def _print_output_epoch(self, e, val_dict):
        it_output = '\n---------------------------------\nFinish epoch: {} \n'.format(e)
        total = 0.
        for k in val_dict.keys():
            dp_mean = val_dict[k] / steps_of_train
            total += dp_mean
            it_output += '\tkey: {}, mean: {:.4f}\n'.format(k, dp_mean)

        it_output += '\tTotal mean: {:.4f}\n'.format(total/len(val_dict.keys()))
        it_output += '\n---------------------------------\n'
        
        return it_output  -->